# M6.A9 — DNN 도입 여부 결정: AutoGluon-TS probe

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A9 · 전환 조건표: `docs/research/ai/00_ml_guide_reference.md` §2.1
> 실행 env: **`sajura-ag`** (autogluon.timeseries 1.5 — 의존성이 무거워 `[ml]`에 넣지 않고 격리, 일회성 probe) · 작성: 2026-07-28

🔒 **공개 저장소 데이터 정책** — 매출 절대액 커밋 금지. 출력 제거 상태로 추적, 본문은 상대값만.

**공정 비교 프로토콜** — V1-t 하네스와 동일 조건:
- fold별 **1회 학습**(train = 검증 월 이전 전체), 검증 월의 **영업일마다** 전일까지의 실측을 컨텍스트로
  **1-step 예측**(teacher forcing) — GBM의 일별 lag 갱신과 동등
- 캘린더 일 단위 시계열, 무매출일 = **NaN**(0 아님 — 수요 왜곡 방지, AG 내부 처리)
- known covariates: 공휴일·개강 직후·학기 주차·기온·재개장 경과 (Chronos는 zero-shot이라 미사용 — 명시)
- test(2026-04) 봉인 유지, 선택 fold 5개만

**사전 고정 판정 기준** — 최고 DNN 계열 모델의 선택 fold 평균 MAE가 **V1-t(05 SSOT) 대비 -5% 이상
개선하면 "도입 재검토"**, 미만이면 **보류 확정**. MA-7보다도 나쁘면 강한 보류 근거.

**재현 노트** — DNN 학습(DeepAR·PatchTST)은 `random_seed` 고정에도 스레드 비결정성으로 수 % 변동 가능.
Chronos-bolt(사전학습 zero-shot)·SeasonalNaive는 결정적.

## 판정 요약 (TL;DR)

1. **판정: DNN 도입 보류 확정** — 최고 DNN 계열(Chronos-bolt zero-shot)의 선택 fold 평균 MAE가
   **V1-t 대비 +8.9% 열세** — 사전 고정 기준(-5% 이상 개선 시 재검토) 미달. `model_spec.md` §3의
   "AutoGluon probe 후 결정" 미확정 해소.
2. **학습형 DNN은 나이브 이하** — DeepAR +34.1%·PatchTST +36.6%로 SeasonalNaive(+28.4%)보다도 열세.
   표본 256일에서 학습형 딥 모델이 이길 수 없다는 사전 가설 그대로 확인.
3. **부가 발견: Chronos-bolt의 잠재력** — 안정 fold 4개에선 V1-t와 동급 수준(2025-12엔 오히려 -19% 우세),
   **regime fold(2026-03)에서만 +44% 붕괴** — zero-shot이라 재개장·학사 covariates를 쓸 수 없는 구조적
   한계가 정확히 급변 구간에서 드러남. MA-7 대비로는 -0.7% 동급 = "MA-7이 이미 강한 기준선"(04 §2) 재확인.
4. **함의 2건** — ① 재평가 트리거에 "Chronos 계열 fine-tuning·covariates 지원 릴리스" 추가
   ② **신규 매장 cold-start 후보**: 이력 없이도 동작하는 zero-shot 특성 — Phase 7+ 설계 메모.
5. **Phase 6 전 마일스톤(M6.A1~A9) 완료** — 남은 것: ai → main 머지 PR + 담당자 검수 3건.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
feat, _ = pp.impute_weather(feat)
ob = feat[feat.is_open]
y = ob.total_amount
folds = pp.make_monthly_folds(ob.index)
SEL = folds[:-1]
CAL = pd.date_range(feat.index.min(), feat.index.max(), freq="D")

base = pd.DataFrame({"item_id": "jochiwon", "timestamp": CAL, "y": y.reindex(CAL).values})
KC = ["is_holiday", "is_semester_first2w", "semester_week", "temp_avg", "days_since_reopen"]
cov = feat[KC].reindex(CAL).astype(float)
cov.index.name = "timestamp"
cov = cov.reset_index()
cov["item_id"] = "jochiwon"
TSDF = TimeSeriesDataFrame.from_data_frame(base.merge(cov, on=["item_id", "timestamp"]),
                                           id_column="item_id", timestamp_column="timestamp")

HP = {"SeasonalNaive": {}, "Chronos": {"model_path": "bolt_small"}, "DeepAR": {}, "PatchTST": {}}
V1T = 296862  # 05 SSOT — V1-t 선택 fold 평균 MAE(원)
V1T_FOLD = {"2025-09": 405496, "2025-10": 154448, "2025-11": 228631, "2025-12": 254982, "2026-03": 440751}
MA7 = 325457
print("프로토콜 준비 완료 — fold:", [f["month"] for f in SEL])

In [ ]:
# fold별 1회 fit → 검증 월 영업일 롤링 1-step (전일 실측 컨텍스트)
results, fit_log = {}, []
for f in SEL:
    m_, tr_end = f["month"], f["train"].max()
    train_tsdf = TSDF.loc[TSDF.index.get_level_values("timestamp") <= tr_end]
    pred = TimeSeriesPredictor(target="y", prediction_length=1, eval_metric="MAE", freq="D",
                               known_covariates_names=KC, verbosity=0,
                               path=str(Path.home() / f".ag_probe/{m_}"))
    t0 = time.time()
    pred.fit(train_tsdf, hyperparameters=HP, time_limit=300, enable_ensemble=False, random_seed=42)
    fitted = pred.model_names()
    fit_log.append(f"[{m_}] fit {time.time()-t0:.0f}s | {fitted}")
    print(fit_log[-1], flush=True)

    per_model = {n: [] for n in fitted}
    for d in f["val"]:
        ctx = TSDF.loc[TSDF.index.get_level_values("timestamp") < d]
        kc = TSDF.loc[TSDF.index.get_level_values("timestamp") == d][KC]
        for n in fitted:
            per_model[n].append(float(pred.predict(ctx, known_covariates=kc, model=n)["mean"].iloc[0]))
    a = y.loc[f["val"]].values
    for n in fitted:
        pv = np.array(per_model[n])
        results.setdefault(n, {})[m_] = (float(np.mean(np.abs(a - pv))),
                                         float(np.mean(2 * np.abs(a - pv) / (a + np.abs(pv))) * 100))
print("롤링 평가 완료", flush=True)

In [ ]:
# 결과 표 — V1-t·MA-7 참조와 비교
rows = []
for n, r in results.items():
    maes = [r[m][0] for m in r]
    smapes = [r[m][1] for m in r]
    rows.append((n, np.mean(maes), f"{np.mean(smapes):.1f}%", f"{np.mean(maes)/V1T-1:+.1%}", f"{np.mean(maes)/MA7-1:+.1%}"))
rows.sort(key=lambda x: x[1])
tbl = pd.DataFrame([(n, f"{v:,.0f}", s, dv, dm) for n, v, s, dv, dm in rows],
                   columns=["모델", "MAE 평균(원)", "sMAPE", "vs V1-t", "vs MA-7"])
display(tbl)

fold_tbl = pd.DataFrame({n: {m: f"{r[m][0]:,.0f}" for m in r} for n, r in results.items()}).T
fold_tbl.loc["V1-t (참조)"] = {m: f"{v:,.0f}" for m, v in V1T_FOLD.items()}
display(fold_tbl)

fig, ax = plt.subplots(figsize=(8.5, 3.6), constrained_layout=True)
names = [r[0] for r in rows] + ["V1-t (참조)"]
vals = [r[1] / 1e4 for r in rows] + [V1T / 1e4]
colors = [PAL["aqua"]] * len(rows) + [PAL["blue"]]
bars = ax.bar(names, vals, color=colors, width=0.55)
ax.axhline(MA7 / 1e4, color=PAL["orange"], lw=1.4, ls="--")
ax.annotate("MA-7 기준선", (len(names) - 0.5, MA7 / 1e4 + 0.4), fontsize=8.5, color=PAL["orange"], ha="right")
ax.set_ylabel("선택 fold 평균 MAE (만원)")
ax.set_title("AutoGluon-TS probe — 동일 하네스 1-step 비교")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="x", visible=False)
plt.show()

### 관찰

- **순위**: Chronos-bolt 323K(+8.9% vs V1-t, -0.7% vs MA-7) ≪ SeasonalNaive(+28.4%) < DeepAR(+34.1%)
  < PatchTST(+36.6%). 학습형 딥 모델 둘 다 계절 나이브보다 못함 — 256일 표본의 한계가 수치로 확정.
- **fold별 스토리가 판정의 핵심**: Chronos는 2025-10/11/12에서 V1-t와 동급~우세(특히 2025-12는 -19% —
  V1-t가 연말 변동을 과보정했던 fold를 사전학습 리듬이 더 잘 탐), 2025-09(개강)도 근소 우세.
  그러나 **2026-03(재개장+개강)에서 636K vs 441K(+44%)** — 업종 개편이라는 이 데이터만의 사건은
  사전학습에 없고 zero-shot은 regime 피처도 못 받는다. V1-t의 우위가 정확히 여기서 나온다는 것이
  하이브리드 채택(04 §3)의 재확인.
- 대규모 사전학습이 "소매 시계열의 보편 리듬(요일·수준 추적)"을 이미 담고 있음은 인상적 — 다만 우리
  문제의 승부처가 보편 리듬이 아니라 **매장 고유 사건(regime·학사)**이라는 점이 결론을 가른다.

## 판정·다음 단계

**M6.A9 종료 판정 — DNN 도입 보류 확정.** 사전 고정 기준(-5%) 미달(+8.9%)이며, research §2.1
전환 조건표 기준으로도 "GBM이 베이스라인 압도 실패" 조건이 성립하지 않는다(V1-t가 유일한 우위 모델).
LSTM/TimExer(구 4·5단계) 별도 실험은 불필요 — 동급 계열(DeepAR·PatchTST)이 이미 나이브 이하.

### spec 반영 (PR #18)

- `model_spec.md` §3 "DNN 계열 도입 여부 및 전환 기준은 AutoGluon 베이스라인 probe 후 결정" → 본 판정으로 해소
- §2 blockquote의 4·5단계(LSTM/TimExer) 문구 → 판정 결과로 갱신
- 재평가 트리거(전환 조건표 연계): ① 데이터 2년+ 축적 ② 다매장 확장(전이 학습 가치) ③ V1-t가 MA-7 대비
  skill을 지속 상실(drift) ④ Chronos 계열 사전학습 모델의 대폭 개선 릴리스

### Phase 6 종료

M6.A1~A9 전체 완료 — 남은 것은 **ai → main 머지 PR**(정책상 Phase 6 마무리 1회)과 담당자 검수 3건.